In [1]:
import csv
import json
import os
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

# ── Optional: rich for pretty tables (graceful fallback if not installed) ──────
try:
    from rich.console import Console
    from rich.table import Table
    from rich import box
    console = Console()
    RICH = True
except ImportError:
    RICH = False

# ╔══════════════════════════════════════════════════════╗
# ║                     CONFIG                          ║
# ╚══════════════════════════════════════════════════════╝
DEMO_MODE    = True                        # False = interactive menu
STORAGE_DIR  = Path("/kaggle/working")     # change to Path(".") for local use
CSV_FILE     = STORAGE_DIR / "expenses.csv"
JSON_FILE    = STORAGE_DIR / "expenses.json"

CATEGORIES = [
    "Food", "Transport", "Housing", "Entertainment",
    "Healthcare", "Education", "Shopping", "Utilities", "Other",
]



@dataclass
class Expense:
    """Represents a single expense entry."""
    id         : int
    date       : str          # ISO format  YYYY-MM-DD
    category   : str
    description: str
    amount     : float
    currency   : str = "USD"

    
    @property
    def month_key(self) -> str:
        """Return  YYYY-MM  for grouping."""
        return self.date[:7]

    @property
    def formatted_amount(self) -> str:
        return f"{self.currency} {self.amount:,.2f}"

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "Expense":
        return cls(
            id          = int(d["id"]),
            date        = d["date"],
            category    = d["category"],
            description = d["description"],
            amount      = float(d["amount"]),
            currency    = d.get("currency", "USD"),
        )

    def __str__(self) -> str:
        return (f"[{self.id:04d}] {self.date}  {self.category:<15} "
                f"{self.formatted_amount:>12}  {self.description}")


@dataclass
class MonthlySummary:
    """Aggregated stats for a calendar month."""
    month      : str
    total      : float = 0.0
    count      : int   = 0
    by_category: dict  = field(default_factory=dict)

    @property
    def average(self) -> float:
        return self.total / self.count if self.count else 0.0

    @property
    def top_category(self) -> str:
        if not self.by_category:
            return "N/A"
        return max(self.by_category, key=self.by_category.get)



class StorageManager:
    """Handles persistence: read/write both CSV and JSON formats."""

    CSV_FIELDS = ["id", "date", "category", "description", "amount", "currency"]

    def __init__(self, csv_path: Path, json_path: Path):
        self.csv_path  = csv_path
        self.json_path = json_path
        csv_path.parent.mkdir(parents=True, exist_ok=True)

    
    def save(self, expenses: list[Expense]) -> None:
        self._save_csv(expenses)
        self._save_json(expenses)

    def _save_csv(self, expenses: list[Expense]) -> None:
        with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=self.CSV_FIELDS)
            writer.writeheader()
            for e in expenses:
                writer.writerow(e.to_dict())

    def _save_json(self, expenses: list[Expense]) -> None:
        data = {
            "last_updated": datetime.now().isoformat(),
            "count"       : len(expenses),
            "expenses"    : [e.to_dict() for e in expenses],
        }
        with open(self.json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    
    def load(self) -> list[Expense]:
        """Prefer JSON (richer metadata); fall back to CSV; return [] if neither exists."""
        if self.json_path.exists():
            return self._load_json()
        if self.csv_path.exists():
            return self._load_csv()
        return []

    def _load_json(self) -> list[Expense]:
        with open(self.json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return [Expense.from_dict(d) for d in data.get("expenses", [])]

    def _load_csv(self) -> list[Expense]:
        expenses = []
        with open(self.csv_path, "r", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                expenses.append(Expense.from_dict(row))
        return expenses

In [2]:
#  EXPENSE TRACKER  (main controller)

class ExpenseTracker:
    """Core application logic — add, delete, query, and report expenses."""

    def __init__(self, storage: StorageManager):
        self.storage  = storage
        self.expenses : list[Expense] = storage.load()
        self._next_id : int           = max((e.id for e in self.expenses), default=0) + 1

    
    def add(
        self,
        amount      : float,
        category    : str,
        description : str,
        date        : Optional[str] = None,
        currency    : str = "USD",
    ) -> Expense:
        """Create and persist a new expense. Returns the saved Expense."""
        if amount <= 0:
            raise ValueError("Amount must be positive.")
        if category not in CATEGORIES:
            raise ValueError(f"Category must be one of: {CATEGORIES}")

        expense = Expense(
            id          = self._next_id,
            date        = date or datetime.today().strftime("%Y-%m-%d"),
            category    = category,
            description = description.strip(),
            amount      = round(float(amount), 2),
            currency    = currency,
        )
        self.expenses.append(expense)
        self._next_id += 1
        self.storage.save(self.expenses)
        return expense

    def delete(self, expense_id: int) -> bool:
        """Remove an expense by ID. Returns True if found and deleted."""
        before = len(self.expenses)
        self.expenses = [e for e in self.expenses if e.id != expense_id]
        if len(self.expenses) < before:
            self.storage.save(self.expenses)
            return True
        return False

    def get_all(self) -> list[Expense]:
        return sorted(self.expenses, key=lambda e: e.date, reverse=True)

    def filter(
        self,
        month     : Optional[str] = None,   # "YYYY-MM"
        category  : Optional[str] = None,
        min_amount: Optional[float] = None,
        max_amount: Optional[float] = None,
    ) -> list[Expense]:
        result = self.expenses
        if month:
            result = [e for e in result if e.month_key == month]
        if category:
            result = [e for e in result if e.category == category]
        if min_amount is not None:
            result = [e for e in result if e.amount >= min_amount]
        if max_amount is not None:
            result = [e for e in result if e.amount <= max_amount]
        return sorted(result, key=lambda e: e.date, reverse=True)

    
    def monthly_summary(self, month: Optional[str] = None) -> list[MonthlySummary]:
        """Return MonthlySummary objects — all months, or a specific one."""
        buckets: dict[str, MonthlySummary] = {}

        for e in self.expenses:
            mk = e.month_key
            if month and mk != month:
                continue
            if mk not in buckets:
                buckets[mk] = MonthlySummary(month=mk)
            s = buckets[mk]
            s.total += e.amount
            s.count += 1
            s.by_category[e.category] = s.by_category.get(e.category, 0) + e.amount

        return sorted(buckets.values(), key=lambda s: s.month, reverse=True)

    def category_totals(self, month: Optional[str] = None) -> dict[str, float]:
        expenses = self.filter(month=month)
        totals: dict[str, float] = defaultdict(float)
        for e in expenses:
            totals[e.category] += e.amount
        return dict(sorted(totals.items(), key=lambda x: x[1], reverse=True))

    @property
    def total_spent(self) -> float:
        return sum(e.amount for e in self.expenses)

In [3]:
#  DISPLAY  (plain-text + optional rich tables)


def _sep(char="═", width=62):
    print(char * width)

def _header(title: str):
    _sep()
    print(f"  {title}")
    _sep()

def display_expenses(expenses: list[Expense], title="Expenses"):
    _header(title)
    if not expenses:
        print("  No expenses found.\n"); return

    if RICH:
        t = Table(box=box.SIMPLE_HEAVY, show_header=True, header_style="bold cyan")
        for col in ("ID", "Date", "Category", "Description", "Amount"):
            t.add_column(col, style="white")
        for e in expenses:
            t.add_row(
                f"{e.id:04d}", e.date, e.category,
                e.description[:40], f"${e.amount:,.2f}"
            )
        console.print(t)
    else:
        print(f"  {'ID':<6} {'Date':<12} {'Category':<15} {'Amount':>10}  Description")
        print("  " + "-" * 58)
        for e in expenses:
            print(f"  {e.id:<6} {e.date:<12} {e.category:<15} "
                  f"${e.amount:>9,.2f}  {e.description[:35]}")
    print()


def display_monthly_summaries(summaries: list[MonthlySummary]):
    _header("Monthly Summaries")
    if not summaries:
        print("  No data available.\n"); return

    if RICH:
        t = Table(box=box.SIMPLE_HEAVY, header_style="bold cyan")
        for col in ("Month", "Expenses", "Total Spent", "Average", "Top Category"):
            t.add_column(col)
        for s in summaries:
            t.add_row(
                s.month, str(s.count),
                f"${s.total:,.2f}", f"${s.average:,.2f}", s.top_category
            )
        console.print(t)
    else:
        print(f"  {'Month':<10} {'#':>5} {'Total':>12} {'Average':>10}  Top Category")
        print("  " + "-" * 55)
        for s in summaries:
            print(f"  {s.month:<10} {s.count:>5} ${s.total:>11,.2f} "
                  f"${s.average:>9,.2f}  {s.top_category}")
    print()


def display_category_report(totals: dict[str, float], label="All Time"):
    _header(f"Spending by Category  ({label})")
    if not totals:
        print("  No data.\n"); return

    grand = sum(totals.values())
    if RICH:
        t = Table(box=box.SIMPLE_HEAVY, header_style="bold cyan")
        t.add_column("Category"); t.add_column("Amount", justify="right")
        t.add_column("Share", justify="right"); t.add_column("Bar")
        for cat, amt in totals.items():
            pct  = amt / grand * 100
            bar  = "█" * int(pct / 5)
            t.add_row(cat, f"${amt:,.2f}", f"{pct:.1f}%", bar)
        console.print(t)
    else:
        print(f"  {'Category':<16} {'Amount':>10}  {'Share':>7}  Bar")
        print("  " + "-" * 52)
        for cat, amt in totals.items():
            pct = amt / grand * 100
            bar = "█" * int(pct / 5)
            print(f"  {cat:<16} ${amt:>9,.2f}  {pct:>6.1f}%  {bar}")
    print()

#  INTERACTIVE MENU


def interactive_menu(tracker: ExpenseTracker):
    MENU = """
  ┌─────────────────────────────┐
  │    EXPENSE TRACKER MENU     │
  ├─────────────────────────────┤
  │  1  Add expense             │
  │  2  View all expenses       │
  │  3  Filter by month         │
  │  4  Category report         │
  │  5  Monthly summaries       │
  │  6  Delete expense          │
  │  7  Total spent             │
  │  0  Exit                    │
  └─────────────────────────────┘
"""
    while True:
        print(MENU)
        choice = input("  Choose an option: ").strip()

        if choice == "1":
            try:
                amount   = float(input("  Amount: $"))
                print(f"  Categories: {', '.join(CATEGORIES)}")
                category = input("  Category: ").strip().title()
                desc     = input("  Description: ").strip()
                date_in  = input("  Date (YYYY-MM-DD, blank=today): ").strip() or None
                e = tracker.add(amount, category, desc, date=date_in)
                print(f"\n  ✔  Added: {e}\n")
            except ValueError as err:
                print(f"\n  ✖  {err}\n")

        elif choice == "2":
            display_expenses(tracker.get_all(), "All Expenses")

        elif choice == "3":
            month = input("  Month (YYYY-MM): ").strip()
            display_expenses(tracker.filter(month=month), f"Expenses for {month}")

        elif choice == "4":
            month = input("  Month (YYYY-MM, blank=all time): ").strip() or None
            display_category_report(
                tracker.category_totals(month),
                label=month or "All Time"
            )

        elif choice == "5":
            display_monthly_summaries(tracker.monthly_summary())

        elif choice == "6":
            try:
                eid = int(input("  Expense ID to delete: "))
                if tracker.delete(eid):
                    print(f"\n  ✔  Expense {eid:04d} deleted.\n")
                else:
                    print(f"\n  ✖  ID {eid:04d} not found.\n")
            except ValueError:
                print("\n  ✖  Invalid ID.\n")

        elif choice == "7":
            print(f"\n  Total spent (all time): ${tracker.total_spent:,.2f}\n")

        elif choice == "0":
            print("\n  Goodbye!\n"); break
        else:
            print("\n  Invalid choice.\n")




DEMO_EXPENSES = [
    # (amount, category, description, date)
    (1200.00, "Housing",       "Monthly rent",               "2024-04-01"),
    (  85.50, "Utilities",     "Electricity bill",           "2024-04-03"),
    (  45.20, "Food",          "Weekly groceries",           "2024-04-05"),
    (  12.99, "Entertainment", "Netflix subscription",       "2024-04-07"),
    (  30.00, "Transport",     "Uber rides",                 "2024-04-10"),
    (  60.00, "Food",          "Restaurant dinner",          "2024-04-14"),
    (  25.00, "Healthcare",    "Pharmacy",                   "2024-04-17"),
    ( 150.00, "Shopping",      "New shoes",                  "2024-04-20"),
    (  40.00, "Education",     "Udemy course",               "2024-04-22"),
    (1200.00, "Housing",       "Monthly rent",               "2024-05-01"),
    (  88.00, "Utilities",     "Electricity + internet",     "2024-05-04"),
    (  52.30, "Food",          "Weekly groceries",           "2024-05-06"),
    (  18.00, "Transport",     "Bus pass",                   "2024-05-09"),
    (  95.00, "Healthcare",    "Doctor visit",               "2024-05-12"),
    (  12.99, "Entertainment", "Netflix subscription",       "2024-05-15"),
    ( 200.00, "Shopping",      "Clothes",                    "2024-05-18"),
    (  55.00, "Food",          "Dinner with friends",        "2024-05-21"),
    (  35.00, "Education",     "Book purchase",              "2024-05-25"),
    (1200.00, "Housing",       "Monthly rent",               "2024-06-01"),
    (  82.00, "Utilities",     "Electricity",                "2024-06-03"),
    (  48.70, "Food",          "Weekly groceries",           "2024-06-06"),
    (  22.00, "Transport",     "Taxi",                       "2024-06-10"),
    (  12.99, "Entertainment", "Netflix subscription",       "2024-06-13"),
    ( 300.00, "Shopping",      "Laptop accessories",         "2024-06-15"),
    (  70.00, "Food",          "Birthday dinner",            "2024-06-20"),
    (  50.00, "Healthcare",    "Gym membership",             "2024-06-22"),
]


def run_demo(tracker: ExpenseTracker):
    print("\n" + "═" * 62)
    print("  EXPENSE TRACKER  —  DEMO MODE")
    print("═" * 62)

    
    print("\n  ── STEP 1: Adding 26 sample expenses …\n")
    for amt, cat, desc, date in DEMO_EXPENSES:
        e = tracker.add(amt, cat, desc, date=date)
        print(f"    ✔  {e}")

    total = sum(e.amount for e in tracker.expenses)
    print(f"\n  Total entries: {len(tracker.expenses)}")
    print(f"  Total spent  : ${total:,.2f}")

    
    display_expenses(tracker.get_all()[:8], "Recent 8 Expenses")

    
    display_expenses(tracker.filter(month="2024-05"), "May 2024 Expenses")

    
    display_category_report(tracker.category_totals(), label="All Time")

    
    display_monthly_summaries(tracker.monthly_summary())

    
    display_category_report(tracker.category_totals(month="2024-06"), label="June 2024")

    
    _header("Delete Expense #3")
    deleted = tracker.delete(3)
    print(f"  Expense 0003 deleted: {deleted}")
    print(f"  Remaining entries : {len(tracker.expenses)}\n")

    print(f"  ✅  Demo complete!")
    print(f"  CSV  → {CSV_FILE}")
    print(f"  JSON → {JSON_FILE}\n")
    print("═" * 62 + "\n")

In [4]:
#  ENTRY POINT

for _f in [CSV_FILE, JSON_FILE]:
    if os.path.exists(_f):
        os.remove(_f)

storage = StorageManager(CSV_FILE, JSON_FILE)
tracker = ExpenseTracker(storage)

if DEMO_MODE:
    run_demo(tracker)
else:
    interactive_menu(tracker)


══════════════════════════════════════════════════════════════
  EXPENSE TRACKER  —  DEMO MODE
══════════════════════════════════════════════════════════════

  ── STEP 1: Adding 26 sample expenses …

    ✔  [0001] 2024-04-01  Housing         USD 1,200.00  Monthly rent
    ✔  [0002] 2024-04-03  Utilities          USD 85.50  Electricity bill
    ✔  [0003] 2024-04-05  Food               USD 45.20  Weekly groceries
    ✔  [0004] 2024-04-07  Entertainment      USD 12.99  Netflix subscription
    ✔  [0005] 2024-04-10  Transport          USD 30.00  Uber rides
    ✔  [0006] 2024-04-14  Food               USD 60.00  Restaurant dinner
    ✔  [0007] 2024-04-17  Healthcare         USD 25.00  Pharmacy
    ✔  [0008] 2024-04-20  Shopping          USD 150.00  New shoes
    ✔  [0009] 2024-04-22  Education          USD 40.00  Udemy course
    ✔  [0010] 2024-05-01  Housing         USD 1,200.00  Monthly rent
    ✔  [0011] 2024-05-04  Utilities          USD 88.00  Electricity + internet
    ✔  [0012] 202

 ID     Date         Category        Description            Amount     
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  0026   2024-06-22   Healthcare      Gym membership         $50.00     
  0025   2024-06-20   Food            Birthday dinner        $70.00     
  0024   2024-06-15   Shopping        Laptop accessories     $300.00    
  0023   2024-06-13   Entertainment   Netflix subscription   $12.99     
  0022   2024-06-10   Transport       Taxi                   $22.00     
  0021   2024-06-06   Food            Weekly groceries       $48.70     
  0020   2024-06-03   Utilities       Electricity            $82.00     
  0019   2024-06-01   Housing         Monthly rent           $1,200.00 


══════════════════════════════════════════════════════════════
  May 2024 Expenses
══════════════════════════════════════════════════════════════


 ID     Date         Category        Description              Amount     
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  0018   2024-05-25   Education       Book purchase            $35.00     
  0017   2024-05-21   Food            Dinner with friends      $55.00     
  0016   2024-05-18   Shopping        Clothes                  $200.00    
  0015   2024-05-15   Entertainment   Netflix subscription     $12.99     
  0014   2024-05-12   Healthcare      Doctor visit             $95.00     
  0013   2024-05-09   Transport       Bus pass                 $18.00     
  0012   2024-05-06   Food            Weekly groceries         $52.30     
  0011   2024-05-04   Utilities       Electricity + internet   $88.00     
  0010   2024-05-01   Housing         Monthly rent             $1,200.00 


══════════════════════════════════════════════════════════════
  Spending by Category  (All Time)
══════════════════════════════════════════════════════════════


...


══════════════════════════════════════════════════════════════
  Monthly Summaries
══════════════════════════════════════════════════════════════


 Month     Expenses   Total Spent   Average   Top Category  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  2024-06   8          $1,785.69     $223.21   Housing       
  2024-05   9          $1,756.29     $195.14   Housing       
  2024-04   9          $1,648.69     $183.19   Housing


══════════════════════════════════════════════════════════════
  Spending by Category  (June 2024)
══════════════════════════════════════════════════════════════


 Category           Amount   Share   Bar            
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Housing         $1,200.00   67.2%   █████████████  
  Shopping          $300.00   16.8%   ███            
  Food              $118.70    6.6%   █              
  Utilities          $82.00    4.6%                  
  Healthcare         $50.00    2.8%                  
  Transport          $22.00    1.2%                  
  Entertainment      $12.99    0.7%


══════════════════════════════════════════════════════════════
  Delete Expense #3
══════════════════════════════════════════════════════════════
  Expense 0003 deleted: True
  Remaining entries : 25

  ✅  Demo complete!
  CSV  → /kaggle/working/expenses.csv
  JSON → /kaggle/working/expenses.json

══════════════════════════════════════════════════════════════

